# 10. Inspect the Iceberg metadata

Opens up the table format: snapshots, manifests, data files, and the real objects sitting in MinIO.

## What to look at

- **snapshots / manifests / files** tables: Iceberg's own bookkeeping, queried with SQL.
- The **object table**: the same chain as physical files: table metadata JSON, manifest list, manifest files, Parquet data files.
- The **metadata JSON summary**: format version, partition spec, sort order.

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
spark = get_spark("nb10")
sql(spark, f"SELECT committed_at, snapshot_id, operation, manifest_list FROM {T_GOLD}.snapshots")

In [ ]:
sql(spark, f"SELECT path, length, added_data_files_count, existing_data_files_count, deleted_data_files_count FROM {T_GOLD}.manifests")

In [ ]:
sql(spark, f"""SELECT file_path, record_count, file_size_in_bytes,
                      readable_metrics.cluster_id.lower_bound AS cluster_min, readable_metrics.cluster_id.upper_bound AS cluster_max
               FROM {T_GOLD}.files ORDER BY file_path LIMIT 8""")

In [ ]:
sql(spark, f"""SELECT add_months(DATE'1970-01-01', partition.purchase_ts_month) AS month, record_count, file_count
               FROM {T_GOLD}.partitions ORDER BY month LIMIT 10""")

In [ ]:
# The same thing as physical objects in MinIO (reuses the helpers from src/10_inspect_iceberg_metadata.py)
ins = load_script("10_inspect_iceberg_metadata.py")
loc = spark.sql(f"DESCRIBE TABLE EXTENDED {T_GOLD}").where("col_name = 'Location'").first().data_type
bucket, _, prefix = loc.removeprefix("s3://").partition("/")
prefix = prefix.rstrip("/") + "/"
objs = ins.list_prefix(ins.s3_client(), bucket, prefix)
def kind(k):
    if k.endswith(".metadata.json"): return "table metadata JSON"
    if "/snap-" in k: return "manifest list"
    if k.endswith(".avro"): return "manifest file"
    if k.endswith(".parquet"): return "data file (Parquet)"
    return "other"
objects = pd.DataFrame([(kind(k), k.removeprefix(prefix), sz) for k, sz in objs], columns=["kind", "object", "bytes"])
print("table location:", loc)
display(objects.groupby("kind").agg(objects=("object", "count"), total_bytes=("bytes", "sum")).reset_index())
objects[objects.kind != "data file (Parquet)"]

In [ ]:
s3 = ins.s3_client()
newest = max((o for k, o in zip(objects.kind, objects.object) if k == "table metadata JSON"))
doc = json.loads(s3.get_object(Bucket=bucket, Key=prefix + newest)["Body"].read())
schema = next(sc for sc in doc["schemas"] if sc["schema-id"] == doc["current-schema-id"])
names = {f["id"]: f["name"] for f in schema["fields"]}
pd.DataFrame([("file", newest), ("format-version", doc["format-version"]),
              ("partition spec", [f"{f['transform']}({names[f['source-id']]})" for f in doc["partition-specs"][-1]["fields"]]),
              ("sort order", [f"{f['transform']}({names[f['source-id']]}) {f['direction']}" for f in doc["sort-orders"][-1]["fields"]]),
              ("snapshots", len(doc.get("snapshots", []))), ("target file size", doc["properties"].get("write.target-file-size-bytes"))],
             columns=["field", "value"])

In [ ]:
# Cleanup. Skip this cell while you are showing the Spark UI (http://localhost:4040): it closes with the session.
stop_spark()

## Open MinIO now

http://localhost:9001 (login `minioadmin` / `minioadmin`), bucket **warehouse**.

Browse `gold/review_embeddings/metadata/`: you can click each object listed above. The `*.metadata.json` is the root, `snap-*.avro` is the manifest list, the other `*.avro` files are manifests. Then `data/` holds the Parquet files the manifests point to. Click a `.metadata.json` and use *Preview* or *Download* to read it.